## Comparative Machine Learning Framework for Clinical Cardiovascular Risk Prediction

This production-ready Python script implements a robust Machine Learning pipeline to predict patient disease risk using clinical parameters. Built on the classic **UCI Heart Disease Dataset**, the project evaluates and compares linear, distance-based, and tree-based ensemble models to identify high-risk cardiovascular profiles.

## 🛠️ System Architecture & Workflow

The script is divided into four distinct operational phases:

### 1. Automated Data Ingestion & Curation

* **Dynamic Fetching:** Programmatically retrieves real-time clinical data directly from the UCI Machine Learning Repository via asynchronous HTTP requests, eliminating manual dependencies.
* **Clinical Imputation:** Detects missing diagnostic records and handles them using statistical properties (Median value imputation for continuous variables like `ca` and Mode imputation for categorical values like `thal`).
* **Target Binarization:** Converts multi-class diagnostic severities into a clean binary classification task ($0 =$ Healthy, $1 =$ High Risk Profile).

### 2. Rigorous Preprocessing & Class Balancing

* **Data Leakage Mitigation:** Splits the dataset into Training ($80\%$) and Testing ($20\%$) subsets using **Stratified Split**, ensuring real-world baseline outcome ratios are preserved in the validation set.
* **Feature Standardization:** Applies `StandardScaler` to normalize numerical parameters (e.g., blood pressure, cholesterol, age), ensuring distance-based estimators converge optimally.
* **SMOTE Synthesis:** Uses **Synthetic Minority Over-sampling Technique (SMOTE)** exclusively on the training pool to artificially generate balanced cohorts, forcing the models to accurately learn minority risk traits.

### 3. Multi-Model Training Engine

The pipeline simultaneously trains and evaluates three distinct classes of machine learning architectures:

* **Logistic Regression:** Establishes a highly interpretable, linear statistical baseline.
* **Support Vector Machine (SVM):** Constructs non-linear, high-dimensional hyperplanes utilizing probability calibration.
* **LightGBM (Light Gradient Boosted Machine):** Employs state-of-the-art tree-based ensemble boosting to detect intricate, multi-variable non-linear dependencies.

### 4. Interactive Presentation Dashboard

Using `plotly`, the script compiles and exports presentation-ready visual analytics:

* **ROC-AUC Comparison Matrix:** An interactive line plot tracking True Positive Rates against False Positive Rates to evaluate absolute discrimination thresholds.
* **Clinical Confusion Matrices:** A side-by-side subplot visualization displaying exact counts of True Positives, True Negatives, False Positives, and critical **False Negatives** (missed diagnoses).


## 📦 Automated Artifact Deliverables

Upon execution, the script automatically writes and triggers direct browser downloads for three core assets:

1. `cleaned_clinical_heart_data.csv`: The polished, imputed, and processed tabular dataset ready for further biostatistical analysis.
2. `dashboard_panel_A_roc.png`: A high-definition, static export of the multi-model ROC evaluation curve.
3. `dashboard_panel_B_confusion_matrix.png`: A clear, visual breakdown of prediction errors, emphasizing model sensitivity and clinical safety.

## 💻 Tech Stack & Dependencies

* **Core Engine:** Python 3.12+
* **Data Engineering:** `pandas`, `numpy`
* **Machine Learning:** `scikit-learn`, `imbalanced-learn`, `lightgbm`
* **Visualization Stack:** `plotly`, `kaleido` (static image engine)

In [17]:
# Cell 1: Import core libraries and load the dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Load the classic UCI Heart Disease Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

df = pd.read_csv(url, names=columns, na_values="?")
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [18]:
# Data Cleaning and Binarization
# 1. Handle missing values (represented as NaN from our read_csv step)
print("Missing values per column before imputation:\n", df.isnull().sum())
df['ca'] = df['ca'].fillna(df['ca'].median())
df['thal'] = df['thal'].fillna(df['thal'].mode()[0])

# 2. Convert target to binary: 0 = Healthy, 1 = Presence of Risk
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

# 3. Check class balance
print("\nClass Distribution:")
print(df['target'].value_counts(normalize=True))

Missing values per column before imputation:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64

Class Distribution:
target
0    0.541254
1    0.458746
Name: proportion, dtype: float64


In [19]:
# Split and Balance Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Split features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale clinical numerical features (crucial for Logistic Regression and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE to training data only to balance the classes synthetically
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Original training shape: {X_train.shape}")
print(f"Resampled training shape: {X_train_res.shape}")

Original training shape: (242, 13)
Resampled training shape: (262, 13)


In [20]:
# Train Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix

# Initialize models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

# Dictionary to hold evaluation data for plotting
model_results = {}

for name, model in models.items():
    # Fit model on balanced data
    model.fit(X_train_res, y_train_res)

    # Predict probabilities and hard labels
    preds_proba = model.predict_proba(X_test_scaled)[:, 1]
    preds = model.predict(X_test_scaled)

    # Save results
    model_results[name] = {
        'preds': preds,
        'proba': preds_proba,
        'model_obj': model
    }

    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, preds))

=== Logistic Regression Classification Report ===
              precision    recall  f1-score   support

           0       0.93      0.82      0.87        33
           1       0.81      0.93      0.87        28

    accuracy                           0.87        61
   macro avg       0.87      0.87      0.87        61
weighted avg       0.88      0.87      0.87        61

=== Support Vector Machine Classification Report ===
              precision    recall  f1-score   support

           0       0.90      0.82      0.86        33
           1       0.81      0.89      0.85        28

    accuracy                           0.85        61
   macro avg       0.85      0.86      0.85        61
weighted avg       0.86      0.85      0.85        61

=== LightGBM Classification Report ===
              precision    recall  f1-score   support

           0       0.94      0.88      0.91        33
           1       0.87      0.93      0.90        28

    accuracy                           0

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



In [21]:
# Interactive ROC-AUC Plot
fig_roc = go.Figure()

for name, results in model_results.items():
    fpr, tpr, thresholds = roc_curve(y_test, results['proba'])
    roc_auc = auc(fpr, tpr)

    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'{name} (AUC = {roc_auc:.3f})',
        hovertemplate='False Positive Rate: %{x:.2f}<br>True Positive Rate: %{y:.2f}<extra></extra>'
    ))

# Add diagonal reference line
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines', line=dict(dash='dash', color='grey'),
    showlegend=False
))

fig_roc.update_layout(
    title='Interactive ROC Curves Comparison',
    xaxis_title='False Positive Rate (1 - Specificity)',
    yaxis_title='True Positive Rate (Sensitivity)',
    template='plotly_white',
    legend=dict(x=0.6, y=0.1)
)

fig_roc.show()

In [22]:
# Interactive Confusion Matrices
from plotly.subplots import make_subplots

fig_cm = make_subplots(rows=1, cols=3, subplot_titles=list(models.keys()))

for i, (name, results) in enumerate(model_results.items(), 1):
    cm = confusion_matrix(y_test, results['preds'])

    # Text labels for the heatmap cells
    z_text = [[str(val) for val in row] for row in cm]

    heatmap = go.Heatmap(
        z=cm,
        x=['Predicted Healthy', 'Predicted At Risk'],
        y=['Actual Healthy', 'Actual At Risk'],
        colorscale='Blues',
        showscale=False,
        text=z_text,
        texttemplate="%{text}",
        textfont={"size": 14}
    )
    fig_cm.add_trace(heatmap, row=1, col=i)

fig_cm.update_layout(
    title_text='Confusion Matrix Comparison Across Models',
    template='plotly_white',
    height=450
)
fig_cm.show()

In [23]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_curve, auc, confusion_matrix

# 1. Fetching Dataset Directly
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]
df = pd.read_csv(url, names=columns, na_values="?")

# 2. Clean & Binarize
df['ca'] = df['ca'].fillna(df['ca'].median())
df['thal'] = df['thal'].fillna(df['thal'].mode()[0])
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

# 3. Split, Scale, and Balance (SMOTE)
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

# 4. Model Training Engine
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

model_results = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    model_results[name] = {
        'preds': model.predict(X_test_scaled),
        'proba': model.predict_proba(X_test_scaled)[:, 1]
    }

# THE INTERACTIVE DASHBOARD CODE

print("\n...Rendering Dashboard Panels...")

# --- PANEL A: INTERACTIVE ROC-AUC CHART ---
fig_roc = go.Figure()
for name, results in model_results.items():
    fpr, tpr, _ = roc_curve(y_test, results['proba'])
    roc_auc = auc(fpr, tpr)

    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{name} (AUC = {roc_auc:.3f})',
        hovertemplate='False Positive Rate: %{x:.2f}<br>True Positive Rate: %{y:.2f}<extra></extra>'
    ))

fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(dash='dash', color='grey'), showlegend=False))
fig_roc.update_layout(
    title='<b>Panel A: Interactive ROC Curves Evaluation</b>',
    xaxis_title='False Positive Rate (1 - Specificity)',
    yaxis_title='True Positive Rate (Sensitivity)',
    template='plotly_white',
    legend=dict(x=0.6, y=0.1)
)
fig_roc.show()

# --- PANEL B: SIDE-BY-SIDE CONFUSION MATRICES ---
fig_cm = make_subplots(rows=1, cols=3, subplot_titles=[f"<b>{m}</b>" for m in models.keys()])
for i, (name, results) in enumerate(model_results.items(), 1):
    cm = confusion_matrix(y_test, results['preds'])
    z_text = [[str(val) for val in row] for row in cm]

    heatmap = go.Heatmap(
        z=cm,
        x=['Predicted Healthy', 'Predicted At Risk'],
        y=['Actual Healthy', 'Actual At Risk'],
        colorscale='Viridis',
        showscale=False,
        text=z_text,
        texttemplate="%{text}",
        textfont={"size": 14}
    )
    fig_cm.add_trace(heatmap, row=1, col=i)

fig_cm.update_layout(
    title_text='<b>Panel B: Clinical Confusion Matrix Comparison</b>',
    template='plotly_white',
    height=450
)
fig_cm.show()

# --- PANEL C: RISK FACTOR DISTRIBUTION CHART ---
df_viz = X_test.copy()
df_viz['Actual Target'] = y_test.map({0: 'Healthy', 1: 'High Risk'})

fig_dist = px.box(
    df_viz,
    x='Actual Target',
    y='age',
    color='Actual Target',
    points="all",
    title="<b>Panel C: Patient Age Distribution vs Clinical Risk Profile</b>",
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_dist.update_layout(template='plotly_white')
fig_dist.show()


...Rendering Dashboard Panels...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

